# Text Generation and Semantic Search

Develop and compare text generation optimization layers: contrast programmatic outputs of Greedy Search, Beam Search, and a Semantic Search index.

In [1]:
import torch
import torch.nn.functional as F

## Setup

In [2]:
vocab = ["<pad>", "I", "love", "learning", "deep", "models", "language"]
vocab_size = len(vocab)
start_token = 1

def get_logits(sequence):
    torch.manual_seed(sum(sequence))
    return torch.randn(vocab_size)

## Greedy Search

In [3]:
def greedy_search(max_len):
    sequence = [start_token]
    for _ in range(max_len):
        logits = get_logits(sequence)
        next_token = torch.argmax(logits).item()
        sequence.append(next_token)
    return [vocab[idx] for idx in sequence]

## Beam Search

In [4]:
def beam_search(max_len, beam_width):
    beams = [([start_token], 0.0)]
    
    for _ in range(max_len):
        all_candidates = []
        for seq, score in beams:
            logits = F.log_softmax(get_logits(seq), dim=0)
            top_logits, top_indices = torch.topk(logits, beam_width)
            
            for i in range(beam_width):
                candidate_seq = seq + [int(top_indices[i].item())]
                candidate_score = score + top_logits[i].item()
                all_candidates.append((candidate_seq, candidate_score))
        
        ordered = sorted(all_candidates, key=lambda x: x[1], reverse=True)
        beams = ordered[:beam_width]
        
    best_seq = beams[0][0]
    return [vocab[idx] for idx in best_seq]

## Semantic Search Index

In [5]:
def semantic_search(query, index_keys, index_values):
    query_tensor = torch.tensor(query, dtype=torch.float32)
    keys_tensor = torch.tensor(index_keys, dtype=torch.float32)
    
    similarities = F.cosine_similarity(query_tensor.unsqueeze(0), keys_tensor)
    best_match_idx = int(torch.argmax(similarities).item())
    
    return index_values[best_match_idx]

## Output Comparison

In [6]:
greedy_output = greedy_search(max_len=4)
beam_output = beam_search(max_len=4, beam_width=3)

corpus_embeddings = [
    [0.1, 0.8, 0.2],
    [0.9, 0.1, 0.5],
    [0.3, 0.4, 0.9]
]
corpus_texts = [
    "I love learning deep models",
    "Language models are fascinating",
    "Semantic search retrieves information"
]
query_embedding = [0.8, 0.2, 0.4]

semantic_output = semantic_search(query_embedding, corpus_embeddings, corpus_texts)

print("Greedy Search Output:  ", " ".join(greedy_output))
print("Beam Search Output:    ", " ".join(beam_output))
print("Semantic Search Output:", semantic_output)

Greedy Search Output:   I <pad> <pad> <pad> <pad>
Beam Search Output:     I learning love language deep
Semantic Search Output: Language models are fascinating
